# DCQO on a compact CVRP QUBO

This notebook uses **CVRP** (unit demand) with a **small quantum layer**:

1. **Classical routing** — builds **K** feasible full solutions; **K** is computed per instance by `compute_min_qubit_budget` (as small as practical — not a fixed constant).
2. **Per-route tours** — brute-force TSP only for small routes; larger routes use **nearest-neighbor** when **C** is large.
3. **One-hot QUBO** — **K** binaries pick one candidate (**K** is minimized for that instance).
4. **BF-DCQO** — Aer `statevector`; **shots** scale gently with **K**.

**Decode** — valid one-hot (min energy), else marginal vote, else cheapest pool entry.


In [ ]:
import itertools
import math
import random
import time
import warnings

import numpy as np

warnings.filterwarnings("ignore")

from collections import defaultdict
from itertools import combinations, permutations, product

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

try:
    from qiskit.transpiler.exceptions import CircuitTooWideForTarget
except Exception:
    CircuitTooWideForTarget = Exception


In [ ]:
# --- Distances & CVRP helpers (unit demand) ---
# One-hot width K = len(pool) <= target_pool_size from compute_min_qubit_budget(...) per instance.

# Above this many stops per vehicle, use fast NN tour instead of brute-force permutations.
MAX_TSP_BRUTEFORCE = 8

# Exact assignment enumeration is O(Nv^n). Skip when too large (heuristic seed instead).
EXACT_MAX_CUSTOMERS = 11
EXACT_MAX_ASSIGNMENTS = 4_000_000  # cap Nv^n


def compute_min_qubit_budget(n_customers, Nv, capacity, demands):
    """Minimal one-hot size K from instance data only — sublinear in n, no global cap."""
    total = float(sum(demands))
    min_routes = int(math.ceil(total / capacity))
    k = 2 + int(math.ceil(math.log2(max(2, n_customers))))
    extra = max(0, Nv - min_routes)
    k += min(2, extra)
    util = total / max(1e-9, Nv * capacity)
    if util > 0.88:
        k += 1
    k = max(3, k)
    k = min(k, n_customers + 1)
    return int(k)


def euclidean_distance(p1, p2):
    return math.hypot(p1[0] - p2[0], p1[1] - p2[1])


def build_distance_matrix(nodes):
    n = len(nodes)
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            d = euclidean_distance(nodes[i], nodes[j])
            D[i, j] = D[j, i] = d
    return D


def route_cost_from_order(D, cust_order):
    depot = 0
    c = D[depot, cust_order[0]]
    for t in range(len(cust_order) - 1):
        c += D[cust_order[t], cust_order[t + 1]]
    c += D[cust_order[-1], depot]
    return float(c)


def tsp_nearest_neighbor(D, subset):
    """O(k^2) tour for a subset of customers when k is large."""
    if not subset:
        return [], 0.0
    unvisited = set(subset)
    cur = min(unvisited, key=lambda j: D[0, j])
    unvisited.remove(cur)
    tour = [cur]
    while unvisited:
        nxt = min(unvisited, key=lambda j: D[cur, j])
        unvisited.remove(nxt)
        tour.append(nxt)
        cur = nxt
    return tour, route_cost_from_order(D, tour)


def best_tsp_on_subset(D, subset):
    if not subset:
        return [], 0.0
    if len(subset) == 1:
        k = subset[0]
        return [k], route_cost_from_order(D, [k])
    if len(subset) > MAX_TSP_BRUTEFORCE:
        return tsp_nearest_neighbor(D, subset)
    best_c = float("inf")
    best_o = None
    for perm in permutations(subset):
        c = route_cost_from_order(D, list(perm))
        if c < best_c:
            best_c, best_o = c, list(perm)
    return best_o, best_c


def routes_from_groups(nodes, D, groups):
    routes = []
    total = 0.0
    for g in groups:
        if not g:
            routes.append([0, 0])
            continue
        order, cst = best_tsp_on_subset(D, g)
        total += cst
        routes.append([0] + order + [0])
    return total, routes


def exact_cvrp(nodes, Nv, C, demands):
    n = len(nodes) - 1
    D = build_distance_matrix(nodes)
    best = float("inf")
    best_routes = None
    for assign in product(range(Nv), repeat=n):
        loads = [0] * Nv
        bad = False
        for i, a in enumerate(assign):
            loads[a] += demands[i]
            if loads[a] > C:
                bad = True
                break
        if bad:
            continue
        groups = [[] for _ in range(Nv)]
        for i, a in enumerate(assign):
            groups[a].append(i + 1)
        tot, routes = routes_from_groups(nodes, D, groups)
        if tot < best:
            best, best_routes = tot, routes
    return float(best), best_routes


def random_feasible_routes(nodes, Nv, C, demands, rng, tries=800):
    n = len(nodes) - 1
    D = build_distance_matrix(nodes)
    for _ in range(tries):
        assign = [rng.randrange(Nv) for _ in range(n)]
        loads = [0] * Nv
        ok = True
        for i, a in enumerate(assign):
            loads[a] += demands[i]
            if loads[a] > C:
                ok = False
                break
        if not ok:
            continue
        groups = [[] for _ in range(Nv)]
        for i, a in enumerate(assign):
            groups[a].append(i + 1)
        return routes_from_groups(nodes, D, groups)
    return None, None


def heuristic_best_random(nodes, Nv, C, demands, rng, total_samples, tries_each):
    """Best of many random feasible solutions — linear cost in total_samples."""
    best_c, best_r = float("inf"), None
    for _ in range(total_samples):
        r = random_feasible_routes(nodes, Nv, C, demands, rng, tries=tries_each)
        if r[0] is None:
            continue
        if r[0] < best_c:
            best_c, best_r = r[0], r[1]
    if best_r is None:
        return None, None
    return best_c, best_r


def swap_customers_routes(routes, Nv, nodes, C, demands, rng, attempts=120):
    D = build_distance_matrix(nodes)
    groups = [[x for x in r if x != 0] for r in routes]
    flat = [(v, idx, c) for v, g in enumerate(groups) for idx, c in enumerate(g)]
    if len(flat) < 2:
        return None, None
    for _ in range(attempts):
        (v1, i1, c1), (v2, i2, c2) = rng.sample(flat, 2)
        if v1 == v2:
            continue
        g1 = groups[v1][:]
        g2 = groups[v2][:]
        g1[i1], g2[i2] = c2, c1
        if sum(demands[x - 1] for x in g1) > C or sum(demands[x - 1] for x in g2) > C:
            continue
        groups2 = [list(x) for x in groups]
        groups2[v1], groups2[v2] = g1, g2
        return routes_from_groups(nodes, D, groups2)
    return None, None


def build_candidate_pool(nodes, Nv, C, demands, target_pool_size, rng, verbose=True):
    D = build_distance_matrix(nodes)
    n = len(nodes) - 1
    enum_size = Nv ** n if n < 20 else float("inf")

    use_exact = n <= EXACT_MAX_CUSTOMERS and enum_size <= EXACT_MAX_ASSIGNMENTS

    if use_exact:
        opt_cost, opt_routes = exact_cvrp(nodes, Nv, C, demands)
        ref_name = "exact"
    else:
        if verbose:
            print(
                f"[candidate pool] exact enumeration skipped (n={n}, Nv^n≈{enum_size:g}); "
                f"using heuristic reference",
                flush=True,
            )
        # Scale samples modestly with n; independent of target_pool_size for asymptotic cost.
        total_samples = min(8000, max(800, 80 * n))
        tries_each = max(30, min(200, 5 * n))
        opt_cost, opt_routes = heuristic_best_random(
            nodes, Nv, C, demands, rng, total_samples=total_samples, tries_each=tries_each
        )
        ref_name = "heuristic_ref"
        if opt_routes is None:
            raise RuntimeError(
                "No feasible solution found (try more vehicles or larger capacity)."
            )

    seen = set()
    pool = []

    def add(cost, routes):
        sig = tuple(tuple(r) for r in routes)
        if sig in seen:
            return
        seen.add(sig)
        pool.append((float(cost), [list(r) for r in routes]))

    add(opt_cost, opt_routes)

    n_seeds = min(100, max(20, 3 * target_pool_size))
    for seed in range(n_seeds):
        r = random_feasible_routes(
            nodes, Nv, C, demands, random.Random(seed), tries=max(40, 3 * n)
        )
        if r[0] is not None:
            add(r[0], r[1])

    base = [opt_routes]
    n_swaps = min(60, 15 + 2 * target_pool_size)
    for rts in base:
        for _ in range(n_swaps):
            alt = swap_customers_routes(rts, Nv, nodes, C, demands, rng)
            if alt[0] is not None:
                add(alt[0], alt[1])

    pool.sort(key=lambda x: x[0])
    uniq = []
    for item in pool:
        if item not in uniq:
            uniq.append(item)
    pool = uniq[:target_pool_size]

    if verbose:
        lo, hi = pool[0][0], pool[-1][0]
        print(
            f"[candidate pool] K={len(pool)} | costs [{lo:.4f}, {hi:.4f}] | reference ({ref_name})={opt_cost:.4f}",
            flush=True,
        )
    return pool, opt_cost, opt_routes, ref_name


In [ ]:
# --- QUBO helpers & BF-DCQO ---

def infer_n(Q):
    if not Q:
        return 0
    return 1 + max(max(i, j) for i, j in Q.keys())


def energy(bits, Q, constant=0.0):
    if isinstance(bits, str):
        n = infer_n(Q)
        x = np.array([int(b) for b in bits[::-1]], dtype=int)
        if len(x) < n:
            x = np.pad(x, (0, n - len(x)))
        x = x[:n]
    else:
        x = np.asarray(bits, dtype=int)
    e = constant
    for (i, j), q in Q.items():
        if i == j:
            e += q * x[i]
        else:
            e += q * x[i] * x[j]
    return float(e)


def build_one_hot_qubo(costs):
    K = len(costs)
    mx, mn = max(costs), min(costs)
    P = mx + max(1.0, mx - mn) + 1.0
    Q = defaultdict(float)
    for i in range(K):
        Q[(i, i)] += costs[i] - P
    for i in range(K):
        for j in range(i + 1, K):
            Q[(i, j)] += 2.0 * P
    const = P
    return dict(Q), const, P


def qubo_to_ising(Q, n):
    h = np.zeros(n)
    J = {}
    c = 0.0
    for (i, j), v in Q.items():
        if i == j:
            c += 0.5 * v
            h[i] += 0.5 * v
        else:
            c += 0.25 * v
            h[i] += 0.25 * v
            h[j] += 0.25 * v
            a, b = (i, j) if i < j else (j, i)
            J[(a, b)] = J.get((a, b), 0.0) + 0.25 * v
    return h, J, c


def schedule_lambda(step, total_steps):
    u = step / total_steps
    return float(np.sin(np.pi * (np.sin(np.pi * u / 2.0) ** 2) / 2.0) ** 2)


def bf_dcqo_circuit(h, J, hb, p):
    n = len(h)
    qc = QuantumCircuit(n, n)
    qc.h(range(n))
    dt = 1.0 / max(1, p)
    for s in range(1, p + 1):
        lam = schedule_lambda(s, p)
        ma = 2.0 * dt * (1.0 - lam)
        for q in range(n):
            qc.rx(ma, q)
        for q in range(n):
            th = 2.0 * dt * (lam * h[q] + hb[q])
            qc.rz(th, q)
        for (i, j), coef in J.items():
            th = 2.0 * dt * lam * coef
            qc.rzz(th, i, j)
    qc.measure(range(n), range(n))
    return qc


def transpile_safe(qc, backend, opt):
    lim = getattr(getattr(backend, "configuration", lambda: None)(), "n_qubits", None)
    if lim is None or qc.num_qubits <= lim:
        try:
            return transpile(qc, backend=backend, optimization_level=opt), True
        except CircuitTooWideForTarget:
            pass
    return transpile(qc, optimization_level=opt), False


def sample(qc, shots, method, opt, label):
    be = AerSimulator(method=method)
    t0 = time.time()
    tc, _ = transpile_safe(qc, be, opt)
    print(
        f"[solver] {label}: transpile {time.time()-t0:.2f}s | qubits={tc.num_qubits} | gates={sum(tc.count_ops().values())}",
        flush=True,
    )
    t1 = time.time()
    ct = be.run(tc, shots=shots).result().get_counts()
    print(f"[solver] {label}: backend {time.time()-t1:.2f}s | unique={len(ct)}", flush=True)
    return ct, sum(tc.count_ops().values())


def elite(counts, Q, const, alpha=0.08):
    rows = []
    for bs, cnt in counts.items():
        rows.extend([(bs, energy(bs, Q, const))] * cnt)
    rows.sort(key=lambda x: x[1])
    k = max(1, int(math.ceil(alpha * len(rows))))
    return rows[:k]


def bias_update(counts, Q, const, n, alpha, signed):
    el = elite(counts, Q, const, alpha=alpha)
    zsum = np.zeros(n)
    best_e = float("inf")
    best_x = None
    for bs, e in el:
        x = np.array([int(b) for b in bs[::-1]], dtype=int)
        if len(x) < n:
            x = np.pad(x, (0, n - len(x)))
        x = x[:n]
        zsum += np.where(x == 0, 1.0, -1.0)
        if e < best_e:
            best_e, best_x = e, x.copy()
    if best_x is None:
        return np.zeros(n), None
    mag = zsum / max(1, len(el))
    if signed:
        return 5.0 * mag, best_x
    direction = np.where(best_x == 0, 1.0, -1.0)
    return direction * np.abs(mag), best_x


def run_bf_dcqo(Q, const, p, bias_iters, shots, alpha, method, opt, label):
    n = infer_n(Q)
    h, J, _ = qubo_to_ising(Q, n)
    hb = np.zeros(n)
    last_counts = {}
    best_e = float("inf")
    best_bs = None
    gates = 0
    t0 = time.time()
    print(f"[solver] {label}: BF-DCQO | vars={n} | p={p} | iters={bias_iters} | shots={shots}", flush=True)
    for it in range(bias_iters):
        qc = bf_dcqo_circuit(h, J, hb, p)
        counts, gc = sample(qc, shots, method, opt, f"{label} it{it+1}/{bias_iters}")
        gates += gc
        last_counts = counts
        for bs in counts:
            e = energy(bs, Q, const)
            if e < best_e:
                best_e = e
                best_bs = bs
        signed = it == bias_iters - 1
        hb, _ = bias_update(counts, Q, const, n, alpha, signed=signed)
    print(f"[solver] {label}: done in {time.time()-t0:.2f}s | best E≈{best_e:.4f} | gates={gates}", flush=True)
    return last_counts, best_bs, best_e, gates, time.time() - t0


def decode_selection(counts, costs, Q, const):
    # Prefer valid one-hot samples (min energy), then marginal vote, else cheapest cost.
    K = len(costs)
    best_idx = None
    best_e = float("inf")
    for bs, _cnt in counts.items():
        x = np.array([int(b) for b in bs[::-1]], dtype=int)
        if len(x) < K:
            x = np.pad(x, (0, K - len(x)))
        x = x[:K]
        if int(x.sum()) != 1:
            continue
        e = energy(bs, Q, const)
        if e < best_e:
            best_e = e
            best_idx = int(np.argmax(x))
    if best_idx is not None:
        return best_idx, "sample_one_hot"
    w = np.zeros(K)
    tot = 0
    for bs, cnt in counts.items():
        x = np.array([int(b) for b in bs[::-1]], dtype=float)
        if len(x) < K:
            x = np.pad(x, (0, K - len(x)))
        w += cnt * x[:K]
        tot += cnt
    if tot > 0 and w.max() > 0:
        return int(np.argmax(w)), "marginal_vote"
    return int(np.argmin(costs)), "fallback_argmin_cost"


In [ ]:
# --- Instances & batch run ---

INSTANCES = {
    1: {"Nv": 2, "C": 5, "nodes": [(0, 0), (-2, 2), (-5, 8), (2, 3)]},
    2: {"Nv": 2, "C": 2, "nodes": [(0, 0), (-2, 2), (-5, 8), (2, 3)]},
    3: {"Nv": 3, "C": 2, "nodes": [(0, 0), (-2, 2), (-5, 8), (2, 3), (5, 7), (2, 4), (2, -3)]},
    4: {
        "Nv": 4,
        "C": 3,
        "nodes": [
            (0, 0), (-2, 2), (-5, 8), (6, 3), (4, 4), (3, 2),
            (0, 2), (-2, 3), (-4, 3), (2, 3), (2, 7), (-2, 5), (-1, 4),
        ],
    },
}

p = 2
bias_iters = 5
alpha = 0.08
simulator_method = "statevector"
optimization_level = 0


def shots_for_k(K):
    """Sublinear in K: bounded simulator load as instances grow."""
    return int(min(1536, max(160, 120 + 40 * K)))


def print_solution(title, nodes, routes):
    D = build_distance_matrix(nodes)
    tot = 0.0
    ri = 1
    for r in routes:
        if len(r) <= 2:
            continue
        d = sum(D[r[t], r[t + 1]] for t in range(len(r) - 1))
        tot += d
        print(f"  {title} r{ri}: {' -> '.join(map(str, r))} | {d:.2f}")
        ri += 1
    print(f"  {title} TOTAL: {tot:.4f}")


def run_instance(iid, rng):
    cfg = INSTANCES[iid]
    nodes = cfg["nodes"]
    Nv, C = cfg["Nv"], cfg["C"]
    dem = [1] * (len(nodes) - 1)
    n_cust = len(nodes) - 1
    target_k = compute_min_qubit_budget(n_cust, Nv, C, dem)
    print(
        f"\n{'#'*60}\nINSTANCE {iid}: {n_cust} customers, {Nv} vehicles, cap {C}\n"
        f"computed K (one-hot / qubits) = {target_k}\n{'#'*60}"
    )

    pool, ref_cost, ref_routes, ref_name = build_candidate_pool(
        nodes, Nv, C, dem, target_k, rng, verbose=True
    )
    costs = [c for c, _ in pool]
    K = len(costs)
    shots = shots_for_k(K)
    Q, const, P = build_one_hot_qubo(costs)

    counts, best_bs, best_e, gates, elapsed = run_bf_dcqo(
        Q, const, p, bias_iters, shots, alpha, simulator_method, optimization_level, f"inst{iid}"
    )
    idx, how = decode_selection(counts, costs, Q, const)
    dcqo_cost, dcqo_routes = pool[idx]
    gap = abs(dcqo_cost - ref_cost)

    ref_label = "exact optimum" if ref_name == "exact" else "classical reference (heuristic)"
    print(f"\n  {ref_label}: {ref_cost:.4f}")
    print_solution("REF", nodes, ref_routes)
    print(f"  DCQO ({how}): {dcqo_cost:.4f}  |  gap vs reference: {gap:.4f}")
    print_solution("DCQO", nodes, dcqo_routes)
    print(f"  Qubits K={K} | shots={shots} | gates={gates} | time={elapsed:.2f}s")

    return {
        "id": iid,
        "K": K,
        "shots": shots,
        "ref": ref_cost,
        "ref_name": ref_name,
        "dcqo": dcqo_cost,
        "gap": gap,
        "gates": gates,
        "time": elapsed,
        "how": how,
    }


rng = random.Random(0)
rows = [run_instance(i, rng) for i in (1, 2, 3, 4)]

print("\n" + "=" * 120)
print(
    f"{'Inst':>5} {'Cust':>5} {'K':>4} {'Shots':>6} {'Ref':>10} {'DCQO':>10} {'Gap':>10} {'Time':>8} {'RefSrc':>10} {'Decode':>16}"
)
print("-" * 120)
for r in rows:
    c = len(INSTANCES[r["id"]]["nodes"]) - 1
    rs = r["ref_name"][:8]
    print(
        f"{r['id']:5d} {c:5d} {r['K']:4d} {r['shots']:6d} {r['ref']:10.4f} {r['dcqo']:10.4f} {r['gap']:10.4f} {r['time']:8.2f} {rs:>10} {r['how']:>16}"
    )


In [ ]:
# --- Run one selected JSON instance through DCQO ---

from pathlib import Path
import json
import random

# Change these inputs when you want to test a different file / instance.
json_instance_file = "/Users/pranaykakkar/Hackathons/QuantumCT-RTRC-qBraid-Challenge/instance_5.json"
json_instance_id = "instance_1"
json_rng_seed = 0

# Optional overrides. Leave as None to use the notebook defaults.
json_override_target_k = None
json_override_shots = None
json_override_p = None
json_override_bias_iters = None
json_verbose_pool = True


def load_dcqo_json_instance(file_path, instance_id):
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"JSON instance file not found: {path}")

    payload = json.loads(path.read_text())
    if not isinstance(payload, list):
        raise ValueError("Expected the JSON file to contain a top-level list of instances")

    cfg = next((item for item in payload if item.get("instance_id") == instance_id), None)
    if cfg is None:
        available = [item.get("instance_id") for item in payload]
        raise ValueError(f"Instance '{instance_id}' not found. Available ids: {available}")

    customers = cfg.get("customers", [])
    if not customers:
        raise ValueError(f"Instance '{instance_id}' has no customers field")

    by_id = {int(c["customer_id"]): c for c in customers}
    if 0 not in by_id:
        raise ValueError("Expected depot with customer_id=0")

    max_id = max(by_id)
    missing = [idx for idx in range(max_id + 1) if idx not in by_id]
    if missing:
        raise ValueError(
            "Expected contiguous customer_id values from 0..max_id for this notebook helper. "
            f"Missing ids: {missing}"
        )

    nodes = [(by_id[idx]["x"], by_id[idx]["y"]) for idx in range(max_id + 1)]
    dem = [int(by_id[idx].get("demand", 1)) for idx in range(1, max_id + 1)]

    return {
        "instance_id": cfg["instance_id"],
        "Nv": int(cfg["Nv"]),
        "C": int(cfg["C"]),
        "nodes": nodes,
        "dem": dem,
    }


def format_instance_header(instance_id):
    if isinstance(instance_id, str) and instance_id.startswith("instance_"):
        suffix = instance_id.split("_", 1)[1]
        if suffix.isdigit():
            return int(suffix)
    return instance_id


def format_solver_label(instance_id):
    if isinstance(instance_id, str) and instance_id.startswith("instance_"):
        suffix = instance_id.split("_", 1)[1]
        return f"inst{suffix}"
    return str(instance_id).replace(" ", "_")


def run_selected_json_instance(
    file_path,
    instance_id,
    rng_seed=0,
    override_target_k=None,
    override_shots=None,
    override_p=None,
    override_bias_iters=None,
    verbose_pool=True,
):
    cfg = load_dcqo_json_instance(file_path, instance_id)
    nodes = cfg["nodes"]
    Nv = cfg["Nv"]
    C = cfg["C"]
    dem = cfg["dem"]
    n_cust = len(nodes) - 1

    target_k = (
        override_target_k
        if override_target_k is not None
        else compute_min_qubit_budget(n_cust, Nv, C, dem)
    )

    header_id = format_instance_header(cfg["instance_id"])
    print(
        f"\n{'#' * 60}\n"
        f"INSTANCE {header_id}: {n_cust} customers, {Nv} vehicles, cap {C}\n"
        f"computed K (one-hot / qubits) = {target_k}\n"
        f"{'#' * 60}"
    )

    rng = random.Random(rng_seed)
    pool, ref_cost, ref_routes, ref_name = build_candidate_pool(
        nodes,
        Nv,
        C,
        dem,
        target_k,
        rng,
        verbose=verbose_pool,
    )

    costs = [cost for cost, _ in pool]
    K = len(costs)
    shots = override_shots if override_shots is not None else shots_for_k(K)
    p_use = override_p if override_p is not None else p
    bias_iters_use = override_bias_iters if override_bias_iters is not None else bias_iters

    solver_label = format_solver_label(cfg["instance_id"])
    counts, best_bs, best_e, gates, elapsed = run_bf_dcqo(
        build_one_hot_qubo(costs)[0],
        build_one_hot_qubo(costs)[1],
        p_use,
        bias_iters_use,
        shots,
        alpha,
        simulator_method,
        optimization_level,
        solver_label,
    )

    Q, const, P = build_one_hot_qubo(costs)
    idx, how = decode_selection(counts, costs, Q, const)
    dcqo_cost, dcqo_routes = pool[idx]
    gap = abs(dcqo_cost - ref_cost)

    ref_label = "exact optimum" if ref_name == "exact" else "classical reference (heuristic)"
    print(f"\n  {ref_label}: {ref_cost:.4f}")
    print_solution("REF", nodes, ref_routes)
    print(f"  DCQO ({how}): {dcqo_cost:.4f}  |  gap vs reference: {gap:.4f}")
    print_solution("DCQO", nodes, dcqo_routes)
    print(f"  Qubits K={K} | shots={shots} | gates={gates} | time={elapsed:.2f}s")

    return {
        "instance_id": cfg["instance_id"],
        "file": str(file_path),
        "Nv": Nv,
        "C": C,
        "n_customers": n_cust,
        "K": K,
        "target_k": target_k,
        "shots": shots,
        "p": p_use,
        "bias_iters": bias_iters_use,
        "reference_cost": ref_cost,
        "reference_source": ref_name,
        "dcqo_cost": dcqo_cost,
        "gap_vs_reference": gap,
        "decode_method": how,
        "gates": gates,
        "time": elapsed,
        "dcqo_routes": dcqo_routes,
        "reference_routes": ref_routes,
    }


json_run_result = run_selected_json_instance(
    file_path=json_instance_file,
    instance_id=json_instance_id,
    rng_seed=json_rng_seed,
    override_target_k=json_override_target_k,
    override_shots=json_override_shots,
    override_p=json_override_p,
    override_bias_iters=json_override_bias_iters,
    verbose_pool=json_verbose_pool,
)
